### BIG - M

In [ ]:
import numpy as np
def big_m_method(c, A, b, constraint_types, objective='min', M=1e7, tol=1e-9, max_iter=200, verbose=False):
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)
    c = np.array(c, dtype=float)
    constraint_types = list(constraint_types)
    m, n = A.shape
    flip_map = {'<=': '>=', '>=': '<=', '=': '='}
    for i in range(m):
        if b[i] < 0:
            A[i] = -A[i]
            b[i] = -b[i]
            constraint_types[i] = flip_map[constraint_types[i]]
    sense = 1.0 if objective == 'min' else -1.0
    c_work = sense * c
    col_names = [f"x{i + 1}" for i in range(n)]
    A_ext_cols = [A[:, k].tolist() for k in range(n)]   # list of columns
    cost_ext = c_work.tolist()
    artificial_cols = []
    for i, ctype in enumerate(constraint_types):
        if ctype == '<=':
            col = [0.0] * m
            col[i] = 1.0
            A_ext_cols.append(col)
            cost_ext.append(0.0)
            col_names.append(f"s{i + 1}")
        elif ctype == '>=':
            col = [0.0] * m
            col[i] = -1.0
            A_ext_cols.append(col)
            cost_ext.append(0.0)
            col_names.append(f"s{i + 1}")

            col2 = [0.0] * m
            col2[i] = 1.0
            A_ext_cols.append(col2)
            cost_ext.append(M)
            col_names.append(f"a{i + 1}")
            artificial_cols.append(len(col_names) - 1)
        elif ctype == '=':
            col = [0.0] * m
            col[i] = 1.0
            A_ext_cols.append(col)
            cost_ext.append(M)
            col_names.append(f"a{i + 1}")
            artificial_cols.append(len(col_names) - 1)
        else:
            raise ValueError("constraint_types entries must be '<=', '>=' or '='")
    A_ext = np.array(A_ext_cols).T
    cost_ext = np.array(cost_ext, dtype=float)
    total_cols = A_ext.shape[1]
    basis = []
    for i, ctype in enumerate(constraint_types):
        name = f"a{i + 1}" if ctype in ('>=', '=') else f"s{i + 1}"
        basis.append(col_names.index(name))
    tableau = np.zeros((m + 1, total_cols + 1))
    tableau[:m, :total_cols] = A_ext
    tableau[:m, -1] = b
    tableau[-1, :total_cols] = cost_ext
    cB = cost_ext[basis]
    z_row = cB @ tableau[:m, :]
    tableau[-1, :] -= z_row
    iterations = 0
    status = 'optimal'
    while True:
        if verbose:
            print(f"\n--- Iteration {iterations} ---")
            print(np.round(tableau, 3))
            print("Basis:", [col_names[b] for b in basis])
        obj_row = tableau[-1, :-1]
        entering = int(np.argmin(obj_row))
        if obj_row[entering] >= -tol:
            break 
        iterations += 1
        if iterations > max_iter:
            status = 'max_iterations_reached'
            break
        col = tableau[:m, entering]
        if np.all(col <= tol):
            status = 'unbounded'
            break
        ratios = np.array([tableau[i, -1] / col[i] if col[i] > tol else np.inf for i in range(m)])
        leaving = int(np.argmin(ratios))
        pivot = tableau[leaving, entering]
        tableau[leaving, :] /= pivot
        for i in range(m + 1):
            if i != leaving:
                tableau[i, :] -= tableau[i, entering] * tableau[leaving, :]
        basis[leaving] = entering
    solution_full = np.zeros(total_cols)
    for i, bcol in enumerate(basis):
        solution_full[bcol] = tableau[i, -1]

    x = solution_full[:n]
    infeasible = any(solution_full[ac] > tol for ac in artificial_cols)
    if infeasible:
        status = 'infeasible'
    obj_value = float(c @ x)
    return {
        'status': status,
        'solution': {col_names[k]: float(x[k]) for k in range(n)},
        'objective_value': obj_value,
        'iterations': iterations,
        'basis_variables': [col_names[k] for k in basis],
    }


if __name__ == "__main__":
    c = [4, 1]
    A = [
        [3, 1],
        [4, 3],
        [1, 2],
    ]
    b = [3, 6, 4]
    constraint_types = ['=', '>=', '<=']
    result = big_m_method(c, A, b, constraint_types, objective='min')
    print("Status:", result['status'])
    print("Solution:", result['solution'])
    print("Objective value:", result['objective_value'])
    print("Iterations:", result['iterations'])

### VAM

In [ ]:
def _balance(supply, demand, cost):
    supply, demand = supply[:], demand[:]
    cost = [row[:] for row in cost]
    total_s, total_d = sum(supply), sum(demand)
    if total_s == total_d:
        return supply, demand, cost, False
    if total_s < total_d:
        supply.append(total_d - total_s)
        cost.append([0.0] * len(demand))
    else:
        demand.append(total_s - total_d)
        for row in cost:
            row.append(0.0)

    return supply, demand, cost, True
def vogel_approximation_method(supply, demand, cost, verbose=True):
    supply, demand, cost, was_balanced_added = _balance(supply, demand, cost)
    m, n = len(supply), len(demand)
    remaining_supply = supply[:]
    remaining_demand = demand[:]
    allocation = [[0.0] * n for _ in range(m)]
    row_done = [False] * m
    col_done = [False] * n
    steps = []
    if was_balanced_added:
        steps.append("Problem was unbalanced -> added a dummy row/column with zero cost.")
    while not all(row_done) and not all(col_done):
        row_penalty = {}
        for i in range(m):
            if row_done[i]:
                continue
            costs = sorted(cost[i][j] for j in range(n) if not col_done[j])
            row_penalty[i] = costs[0] if len(costs) == 1 else costs[1] - costs[0]
        col_penalty = {}
        for j in range(n):
            if col_done[j]:
                continue
            costs = sorted(cost[i][j] for i in range(m) if not row_done[i])
            col_penalty[j] = costs[0] if len(costs) == 1 else costs[1] - costs[0]
        best_row_i, best_row_p = max(row_penalty.items(), key=lambda kv: kv[1])
        best_col_j, best_col_p = max(col_penalty.items(), key=lambda kv: kv[1])
        if best_row_p >= best_col_p:
            i = best_row_i
            j = min((j for j in range(n) if not col_done[j]), key=lambda j: cost[i][j])
        else:
            j = best_col_j
            i = min((i for i in range(m) if not row_done[i]), key=lambda i: cost[i][j])
        qty = min(remaining_supply[i], remaining_demand[j])
        allocation[i][j] += qty
        steps.append(
            f"Allocate {qty} units to cell (row {i + 1}, col {j + 1}), cost/unit = {cost[i][j]} "
            f"[row penalty={row_penalty.get(i)}, col penalty={col_penalty.get(j)}]")
        remaining_supply[i] -= qty
        remaining_demand[j] -= qty
        if remaining_supply[i] == 0:
            row_done[i] = True
        if remaining_demand[j] == 0:
            col_done[j] = True
    total_cost = sum(allocation[i][j] * cost[i][j] for i in range(m) for j in range(n))
    if verbose:
        for s in steps:
            print(s)
        print("\nAllocation matrix:")
        for row in allocation:
            print(row)
        print("Total cost:", total_cost)
    return allocation, total_cost, steps
if __name__ == "__main__":
    supply = [7, 9, 18]
    demand = [5, 8, 7, 14]
    cost = [
        [19, 30, 50, 10],
        [70, 30, 40, 60],
        [40, 8, 70, 20],
    ]
    vogel_approximation_method(supply, demand, cost)

### MODI

In [ ]:
def _fix_degeneracy(allocation, cost, eps=1e-6):
    m, n = len(allocation), len(allocation[0])
    basic_cells = [(i, j) for i in range(m) for j in range(n) if allocation[i][j] != 0]
    needed = m + n - 1
    if len(basic_cells) >= needed:
        return allocation, basic_cells
    parent = list(range(m + n))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    def union(x, y):
        rx, ry = find(x), find(y)
        if rx == ry:
            return False
        parent[rx] = ry
        return True
    for (i, j) in basic_cells:
        union(i, m + j)
    candidates = sorted(
        ((i, j) for i in range(m) for j in range(n) if allocation[i][j] == 0),
        key=lambda cell: cost[cell[0]][cell[1]],
    )
    for (i, j) in candidates:
        if len(basic_cells) >= needed:
            break
        if union(i, m + j):
            allocation[i][j] = eps
            basic_cells.append((i, j))

    return allocation, basic_cells


def _find_loop(candidates, start):
    candidates = set(candidates)
    def search(path, visited, move_along_row):
        current = path[-1]
        if move_along_row:
            options = [c for c in candidates if c[0] == current[0] and c != current]
        else:
            options = [c for c in candidates if c[1] == current[1] and c != current]
        for nxt in options:
            if nxt == start and len(path) >= 3:
                return path
            if nxt in visited:
                continue
            visited.add(nxt)
            path.append(nxt)
            result = search(path, visited, not move_along_row)
            if result is not None:
                return result
            path.pop()
            visited.remove(nxt)
        return None
    for first_move_along_row in (True, False):
        result = search([start], {start}, first_move_along_row)
        if result is not None:
            return result
    return None


def modi_method(cost, allocation, verbose=True, tol=1e-9):
    m, n = len(cost), len(cost[0])
    allocation = [row[:] for row in allocation]
    allocation, basic_cells = _fix_degeneracy(allocation, cost)

    iteration = 0
    while True:
        basic_set = set(basic_cells)
        u = [None] * m
        v = [None] * n
        u[0] = 0
        changed = True
        while changed:
            changed = False
            for (i, j) in basic_set:
                if u[i] is not None and v[j] is None:
                    v[j] = cost[i][j] - u[i]
                    changed = True
                elif v[j] is not None and u[i] is None:
                    u[i] = cost[i][j] - v[j]
                    changed = True
        best = None
        for i in range(m):
            for j in range(n):
                if (i, j) in basic_set:
                    continue
                d = cost[i][j] - (u[i] + v[j])
                if best is None or d < best[0]:
                    best = (d, i, j)

        if best is None or best[0] >= -tol:
            break

        iteration += 1
        _, ei, ej = best
        loop = _find_loop(basic_set | {(ei, ej)}, (ei, ej))
        if loop is None:
            raise RuntimeError("Could not find a closed loop - the basic solution may be degenerate "
                                "in a way this simple implementation does not handle.")

        plus_cells = loop[0::2]
        minus_cells = loop[1::2]
        theta = min(allocation[i][j] for (i, j) in minus_cells)

        for (i, j) in plus_cells:
            allocation[i][j] += theta
        for (i, j) in minus_cells:
            allocation[i][j] -= theta

        leaving = next((i, j) for (i, j) in minus_cells if abs(allocation[i][j]) < 1e-9)
        allocation[leaving[0]][leaving[1]] = 0.0
        basic_cells = [c for c in basic_cells if c != leaving] + [(ei, ej)]

        if verbose:
            print(f"Iteration {iteration}: entering ({ei+1},{ej+1}) [d={best[0]:.3f}], "
                  f"leaving ({leaving[0]+1},{leaving[1]+1}), theta={theta}")

    total_cost = sum(allocation[i][j] * cost[i][j] for i in range(m) for j in range(n))

    if verbose:
        print("\nOptimal allocation:")
        for row in allocation:
            print([round(x, 4) for x in row])
        print("Optimal total cost:", total_cost)

    return allocation, total_cost, iteration


if __name__ == "__main__":
    from vam_method import vogel_approximation_method

    supply = [7, 9, 18]
    demand = [5, 8, 7, 14]
    cost = [
        [19, 30, 50, 10],
        [70, 30, 40, 60],
        [40, 8, 70, 20],
    ]

    initial_allocation, initial_cost, _ = vogel_approximation_method(supply, demand, cost, verbose=False)
    print("VAM initial cost:", initial_cost)

    optimal_allocation, optimal_cost, iters = modi_method(cost, initial_allocation)
    print("\nMODI iterations:", iters)